# My first FFN 

# Classification 
---
Types of classification: https://www.geeksforgeeks.org/machine-learning/getting-started-with-classification/

---

- Binary Classification

- Loading our binary classification data

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("Datasets/heart.csv")

data.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [3]:
# Seperating features and target as X and Y
X = data.drop("HeartDisease", axis=1)
y = data["HeartDisease"]

X.shape, y.shape



((918, 11), (918,))

In [4]:
X

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up
...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat


In [5]:
y

0      0
1      1
2      0
3      1
4      0
      ..
913    1
914    1
915    1
916    1
917    0
Name: HeartDisease, Length: 918, dtype: int64

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
col = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

X[col] = X[col].apply(le.fit_transform)

X.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,1,1,140,289,0,1,172,0,0.0,2
1,49,0,2,160,180,0,1,156,0,1.0,1
2,37,1,1,130,283,0,2,98,0,0.0,2
3,48,0,0,138,214,0,1,108,1,1.5,1
4,54,1,2,150,195,0,1,122,0,0.0,2


- Train test split data (80% training, 20% testing)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((734, 11), (184, 11), (734,), (184,))

In [8]:
# Convert data to tensors

import torch

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

- Building classification model 

In [9]:
import torch
import torch.nn as nn

In [10]:
model = nn.Sequential(
    # Layer 1
    nn.Linear(11, 22),
    nn.ReLU(),

    # Layer 2
    nn.Linear(22, 44),
    nn.ReLU(),

    # Layer 3
    nn.Linear(44, 22),
    nn.ReLU(),

    # Layer 4
    nn.Linear(22, 1),
    nn.Sigmoid()
)

print(model)

Sequential(
  (0): Linear(in_features=11, out_features=22, bias=True)
  (1): ReLU()
  (2): Linear(in_features=22, out_features=44, bias=True)
  (3): ReLU()
  (4): Linear(in_features=44, out_features=22, bias=True)
  (5): ReLU()
  (6): Linear(in_features=22, out_features=1, bias=True)
  (7): Sigmoid()
)


In [11]:
single_layer_example = nn.Linear(11, 22)

In [12]:
single_layer_example(X_train_tensor).shape

torch.Size([734, 22])

In [13]:
# Now decide out loss function for binary classification task

loss_function = nn.BCELoss()

# decide the optimizer we will be using
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [14]:
from sklearn.metrics import accuracy_score

In [15]:
# training loop for our model

for epoch in range(101):
    model.train() # Good practice
    out = model(X_train_tensor)
    loss = loss_function(out, y_train_tensor.unsqueeze(dim=1).float())

    optimizer.zero_grad() # Clears old gradients, NOT weights
    loss.backward()
    optimizer.step()

    # Evaluation Mode
    model.eval() 
    with torch.no_grad(): # Saves memory/time during testing
        y_pred = model(X_test_tensor)
        y_pred_class = (y_pred > 0.5).int()

    # Now calculate accuracy
    accuracy = accuracy_score(y_test_tensor.unsqueeze(dim=1).float().detach().numpy(), y_pred_class.detach().numpy())

    if epoch % 10 == 0:
        print("Current epoch: ", epoch, "| And current loss: ", loss.item(), "| Model Accuracy: ", accuracy)

Current epoch:  0 | And current loss:  2.2345938682556152 | Model Accuracy:  0.5597826086956522
Current epoch:  10 | And current loss:  0.6014321446418762 | Model Accuracy:  0.6032608695652174
Current epoch:  20 | And current loss:  0.5471927523612976 | Model Accuracy:  0.6956521739130435
Current epoch:  30 | And current loss:  0.5254077911376953 | Model Accuracy:  0.6956521739130435
Current epoch:  40 | And current loss:  0.5060530304908752 | Model Accuracy:  0.7282608695652174
Current epoch:  50 | And current loss:  0.4809783697128296 | Model Accuracy:  0.7391304347826086
Current epoch:  60 | And current loss:  0.49641597270965576 | Model Accuracy:  0.7608695652173914
Current epoch:  70 | And current loss:  0.4378516972064972 | Model Accuracy:  0.8043478260869565
Current epoch:  80 | And current loss:  0.40158504247665405 | Model Accuracy:  0.8315217391304348
Current epoch:  90 | And current loss:  0.4894680380821228 | Model Accuracy:  0.7228260869565217
Current epoch:  100 | And cur

# Classic Pytorch Model(Recommended) # IMP

In [20]:
# Example class creation - demo practice
class ExampleClass:
    def __init__(self):
        print("Hello")
        # Use of self key on variable
        self.a = 1
        print(self.a)
        
    # Method
    def forward(self, name):
        print("Your name is:", name)
        # Update variable having self key
        self.a = 100
        print(self.a)
        

# object = class()
a = ExampleClass()

# calling a method for class ExampleClass with name as input parameter
a.forward(name="Aish")


Hello
1
Your name is: Aish
100


- To create a Deep learning AI model in pytorch we use classes.

In [22]:
class BCModel(nn.Module):
    def __init__(self):
        super(BCModel, self).__init__() # Calling nn.Module constructor from BCModel constructor

        # Model layer 
        self.fc1 = nn.Linear(11, 22)
        self.fc2 = nn.Linear(22, 44)
        self.fc3 = nn.Linear(44, 22)
        self.fc4 = nn.Linear(22, 1)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.fc2(out)
        out = self.fc3(out)
        out = self.fc4(out)
        out = self.sigmoid(out)

        return out
        

In [23]:
# object = class name
my_model = BCModel()
my_model

BCModel(
  (fc1): Linear(in_features=11, out_features=22, bias=True)
  (fc2): Linear(in_features=22, out_features=44, bias=True)
  (fc3): Linear(in_features=44, out_features=22, bias=True)
  (fc4): Linear(in_features=22, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [26]:
# loss function for Binary Classification - Binary Cross Entropy
loss_function = nn.BCELoss()


In [28]:
# Optimizer - Adam
optimizer = torch.optim.Adam(my_model.parameters(), lr=0.001)

In [32]:
# training Binary Classification Model

epochs = 101

for epoch in range(epochs):
    # put model in training mode 
    my_model.train()

    # y_train target for X_train
    y_target = y_train_tensor.unsqueeze(dim=1).float()

    # predict value
    out = my_model(X_train_tensor)
    # calculate loss between predicted value and actual value of input x
    loss = loss_function(out, y_target)

    # now adjust the values of weights and biases in neurons using optimizer and loss
    optimizer.zero_grad()
    loss.backward() # do back-propogation
    optimizer.step() # update the weights

    if epoch % 10 == 0:
        print("Epochs: ", epoch, "| Loss: ", loss.item())
        
    

Epochs:  0 | Loss:  0.5593249797821045
Epochs:  10 | Loss:  0.556023895740509
Epochs:  20 | Loss:  0.5527188181877136
Epochs:  30 | Loss:  0.5493873953819275
Epochs:  40 | Loss:  0.5459330081939697
Epochs:  50 | Loss:  0.5423024892807007
Epochs:  60 | Loss:  0.5384389162063599
Epochs:  70 | Loss:  0.5342851877212524
Epochs:  80 | Loss:  0.5297858119010925
Epochs:  90 | Loss:  0.5248863101005554
Epochs:  100 | Loss:  0.5195345282554626
